# NSL-KDD exploratory analysis
Run after `python scripts/download_data.py`. This notebook reads the official training split; it does not retrain or evaluate the model.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root))
from src.data import TRAIN_PATH, load_nsl_kdd, MODEL_FEATURES  # noqa: E402
train = load_nsl_kdd(TRAIN_PATH)
print(train.shape)
train.head()

## Class balance
The family labels are descriptive; the classifier is binary.


In [ ]:
display(train['attack_family'].value_counts().rename('rows').to_frame())
train['attack_family'].value_counts().plot.bar(title='NSL-KDD KDDTrain+ class balance')
plt.ylabel('Rows')
plt.show()

## Key feature distributions
A log transform keeps the long byte-count tails readable.


In [ ]:
sample = train.sample(n=min(10000, len(train)), random_state=42)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feature in zip(axes, ['src_bytes', 'dst_bytes', 'duration']):
    for label, group in sample.groupby('label_binary'):
        ax.hist(np.log1p(group[feature]), bins=40, alpha=0.5, label='Anomaly' if label else 'Normal')
    ax.set_title(f'log1p({feature})')
    ax.legend()
plt.tight_layout()
plt.show()

## Numeric feature correlations
Pearson correlations are descriptive associations, not causal explanations.


In [ ]:
numeric = train[MODEL_FEATURES].select_dtypes(include='number')
corr = numeric.corr()
top = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().nlargest(12)
display(top.rename('absolute Pearson correlation').to_frame())